# Week 5 — Deterministic Rules Baseline & Object-Oriented Random Forest (Binary + Multi-Class)

**Topic 04 (NT219 / NT2205):** *Agentic ML/DL for Metadata-Only TLS/SSH Threat Detection, Hunting & Response*

### Design & Compliance Checklist
- **Strict No-L7 Payload (`SR-01`, `CR-06`)**: Validated automatically before training; operates strictly on 40 L3/L4 features.
- **Anti-Leakage Split (`SR-08`, Section 14)**: Preserves Temporal + Host Holdout + Unseen Attack Holdout; scalers fit strictly on Train.
- **Kaggle GPU / Local Dual-Path**: Automatically reads `/kaggle/input/datasets/vxdwkhoa/Dataset_NF/NF-UQ-NIDS-v2/` on Kaggle, falling back to `D:\Crypto\DoAn\NF-UQ-NIDS-v2` or `nfuq_sample_1500.parquet` locally.
- **OOP Cell Architecture**: Config/PathResolver -> DataLoader/Privacy -> Transformer -> Dataset -> Rules & RF Models -> Trainer -> Evaluator & CheckpointManager.

## Cell 1 — Configuration, Path Resolution & Reproducibility

In [1]:
import json
import math
import os
import platform
import random
import re
import sys
import time
import warnings
from collections.abc import Mapping
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, FrozenSet, List, Optional, Sequence, Tuple

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch

warnings.filterwarnings("ignore", category=UserWarning)

L3_L4_FEATURE_COLUMNS: List[str] = [
    "src_port_class", "dst_port", "proto", "in_bytes", "in_pkts", "out_bytes", "out_pkts",
    "tcp_flags", "client_tcp_flags", "server_tcp_flags", "flow_duration_ms", "duration_in",
    "duration_out", "min_ttl", "max_ttl", "longest_flow_pkt", "shortest_flow_pkt",
    "min_ip_pkt_len", "max_ip_pkt_len", "src_to_dst_second_bytes", "dst_to_src_second_bytes",
    "retransmitted_in_bytes", "retransmitted_in_pkts", "retransmitted_out_bytes",
    "retransmitted_out_pkts", "src_to_dst_avg_throughput", "dst_to_src_avg_throughput",
    "num_pkts_up_to_128_bytes", "num_pkts_128_to_256_bytes", "num_pkts_256_to_512_bytes",
    "num_pkts_512_to_1024_bytes", "num_pkts_1024_to_1514_bytes", "tcp_win_max_in",
    "tcp_win_max_out", "icmp_type", "icmp_ipv4_type", "is_tls_port", "is_ssh_port",
    "bytes_ratio", "pkts_ratio",
]

KAGGLE_DATASET_DIR = Path("/kaggle/input/datasets/vxdwkhoa/Dataset_NF/NF-UQ-NIDS-v2")
KAGGLE_CHECKPOINT_DIR = Path("/kaggle/working/Checkpoint")
LOCAL_DATASET_DIR = Path(r"D:\Crypto\DoAn\NF-UQ-NIDS-v2")
LOCAL_CHECKPOINT_DIR = Path(r"D:\Crypto\DoAn\Notebook\Checkpoint")
REPO_SAMPLE_PATH = Path(r"D:\Crypto\DoAn\topic04-agentic-tls-ssh-threat-detection\data\public_loaders\sample_data\nfuq_sample_1500.parquet")


@dataclass
class ExperimentConfig:
    seed: int = 42
    max_train_rows: Optional[int] = None  # Set e.g. 50000 for fast local tests; None for full Kaggle GPU
    dataset_dir: Optional[str] = None
    checkpoint_dir: Optional[str] = None
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


class PathResolver:
    """Dual-environment resolver: Explicit/ENV -> Kaggle GPU (+ slug scan) -> Local Full -> Repo Sample."""

    @staticmethod
    def _scan_kaggle_input(kaggle_root: Path = Path("/kaggle/input"), max_depth: int = 4) -> Optional[Path]:
        if not kaggle_root.is_dir():
            return None
        queue: List[Tuple[Path, int]] = [(kaggle_root, 0)]
        while queue:
            current, depth = queue.pop(0)
            if (current / "train" / "flows_tabular.parquet").is_file():
                return current
            if depth < max_depth:
                try:
                    children = sorted((p for p in current.iterdir() if p.is_dir()), key=lambda p: p.name)
                except OSError:
                    children = []
                for child in children:
                    queue.append((child, depth + 1))
        return None

    @classmethod
    def resolve(cls, config: ExperimentConfig) -> Dict[str, Any]:
        env_data = os.environ.get("TOPIC04_DATA_DIR", "").strip()
        env_ckpt = os.environ.get("TOPIC04_CHECKPOINT_DIR", "").strip()

        resolved_data: Optional[Path] = None
        mode = "unknown"
        is_sample = False

        # Resolve dataset path
        candidates: List[Tuple[str, Path]] = []
        if config.dataset_dir:
            candidates.append(("custom", Path(config.dataset_dir)))
        elif env_data:
            candidates.append(("custom", Path(env_data)))
        else:
            candidates.append(("kaggle", KAGGLE_DATASET_DIR))

        def _inspect(cand: Path) -> Optional[Tuple[Path, bool]]:
            if not cand.exists():
                return None
            if cand.is_file() and cand.suffix.lower() == ".parquet":
                return cand, True
            if cand.is_dir():
                if (cand / "train" / "flows_tabular.parquet").is_file():
                    return cand, False
                if (cand / "NF-UQ-NIDS-v2" / "train" / "flows_tabular.parquet").is_file():
                    return cand / "NF-UQ-NIDS-v2", False
                if (cand / "nfuq_sample_1500.parquet").is_file():
                    return cand / "nfuq_sample_1500.parquet", True
            return None

        for label, cand in candidates:
            hit = _inspect(cand)
            if hit is not None:
                resolved_data, is_sample = hit
                mode = label
                break

        if resolved_data is None and not config.dataset_dir and not env_data:
            kaggle_input_root = Path("/kaggle/input")
            if kaggle_input_root.exists():
                slug_dir = cls._scan_kaggle_input(kaggle_input_root, max_depth=4)
                if slug_dir is not None:
                    resolved_data, mode, is_sample = slug_dir, "kaggle_slug_scan", False

        if resolved_data is None and not config.dataset_dir and not env_data:
            for label, cand in (
                ("local_full", LOCAL_DATASET_DIR),
                ("local_sample", REPO_SAMPLE_PATH),
                ("local_sample_rel", Path("data/public_loaders/sample_data/nfuq_sample_1500.parquet")),
                ("local_sample_up", Path("../data/public_loaders/sample_data/nfuq_sample_1500.parquet")),
            ):
                hit = _inspect(cand)
                if hit is not None:
                    resolved_data, is_sample = hit
                    mode = label
                    break

        if resolved_data is None:
            raise FileNotFoundError("Could not locate NF-UQ-NIDS-v2 dataset (Kaggle, local, or sample).")

        # Resolve checkpoint directory
        if config.checkpoint_dir:
            ckpt_dir = Path(config.checkpoint_dir)
        elif env_ckpt:
            ckpt_dir = Path(env_ckpt)
        elif Path("/kaggle/working").exists():
            ckpt_dir = KAGGLE_CHECKPOINT_DIR
        elif LOCAL_CHECKPOINT_DIR.parent.exists():
            ckpt_dir = LOCAL_CHECKPOINT_DIR
        else:
            ckpt_dir = Path.cwd() / "Checkpoint"

        ckpt_dir.mkdir(parents=True, exist_ok=True)
        return {
            "mode": mode,
            "is_sample_mode": is_sample,
            "dataset_path": resolved_data,
            "checkpoint_dir": ckpt_dir,
        }


def set_global_seed(seed: int = 42) -> Dict[str, str]:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    return {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__,
        "torch": torch.__version__,
        "joblib": joblib.__version__,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }


CONFIG = ExperimentConfig(seed=42)
VERSIONS = set_global_seed(CONFIG.seed)
PATHS = PathResolver.resolve(CONFIG)
print("Resolved Paths:", {k: str(v) for k, v in PATHS.items()})
print("Reproducibility & Library Versions:", json.dumps({"seed": CONFIG.seed, **VERSIONS}, indent=2))

Resolved Paths: {'mode': 'kaggle_slug_scan', 'is_sample_mode': 'False', 'dataset_path': '/kaggle/input/datasets/vxdwkhoa/dataset-nf/NF-UQ-NIDS-v2', 'checkpoint_dir': '/kaggle/working/Checkpoint'}
Reproducibility & Library Versions: {
  "seed": 42,
  "python": "3.13.18",
  "numpy": "2.1.3",
  "pandas": "2.3.3",
  "scikit-learn": "1.6.1",
  "torch": "2.11.0+cpu",
  "joblib": "1.6.0",
  "device": "cpu"
}


## Cell 2 — Data Loader & No-L7 Privacy Validator

In [2]:
UNSEEN_ATTACK_FAMILIES: FrozenSet[str] = frozenset({"Worms", "Shellcode", "Ransomware"})


class PrivacyAndDataLoader:
    """
    Loads NF-UQ-NIDS-v2 Temporal + Host Holdout + Unseen Attack Holdout splits
    and enforces strict No-L7 Payload privacy validation (SR-01, CR-06, Section 13.4 Step 7).
    """

    FORBIDDEN_PATTERNS = [
        r"^http_", r"^ssh_command", r"^ssh_terminal", r"^ssh_keystroke", r"^ssh_auth_password",
        r"^decrypted_", r"^tls_session_key", r"^private_key", r"^raw_pcap", r"^payload",
        r"^l7_proto$", r"^dns_query_", r"^dns_ttl_", r"^ftp_command_",
        r"^ipv4_src_addr$", r"^ipv4_dst_addr$", r"^raw_s(ni|an)$",
    ]
    IPV4_RE = re.compile(r"^(?:[0-9]{1,3}\.){3}[0-9]{1,3}$")

    def __init__(self, paths: Dict[str, Any]) -> None:
        self.paths = paths

    def validate_dataframe_privacy(self, df: pd.DataFrame, split_name: str) -> Dict[str, Any]:
        violations = []
        for col in df.columns:
            col_l = col.lower()
            for pat in self.FORBIDDEN_PATTERNS:
                if re.search(pat, col_l):
                    violations.append(f"Forbidden L7/raw-IP column detected: {col}")
        for tok_col in ("src_host_token", "dst_host_token"):
            if tok_col in df.columns:
                unique_vals = df[tok_col].dropna().astype(str).unique()
                if any(self.IPV4_RE.match(v) for v in unique_vals):
                    violations.append(f"Raw IPv4 leaked inside {tok_col}")
        if violations:
            raise ValueError(f"Privacy Validator FAILED on {split_name}: {violations}")

        missing_features = [c for c in L3_L4_FEATURE_COLUMNS if c not in df.columns]
        if missing_features:
            raise ValueError(f"Dataset '{split_name}' missing required L3/L4 features: {missing_features}")

        return {"split": split_name, "passed_no_l7_audit": True, "rows": len(df), "violations": 0}

    @staticmethod
    def _sort_by_flow_order(df: pd.DataFrame) -> pd.DataFrame:
        if "flow_order_id" in df.columns and len(df) > 0:
            return df.sort_values("flow_order_id", kind="mergesort").reset_index(drop=True)
        return df.reset_index(drop=True)

    @classmethod
    def _split_sample_temporal(
        cls, df_raw: pd.DataFrame
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        if "attack_family" in df_raw.columns:
            unseen_mask = df_raw["attack_family"].astype(str).isin(UNSEEN_ATTACK_FAMILIES)
        else:
            unseen_mask = pd.Series(False, index=df_raw.index)

        df_unseen = cls._sort_by_flow_order(df_raw.loc[unseen_mask].copy())
        df_known = cls._sort_by_flow_order(df_raw.loc[~unseen_mask].copy())

        n = len(df_known)
        if n == 0:
            df_train = df_known.copy()
            df_val = df_known.iloc[0:0].copy()
            df_test_known = df_known.iloc[0:0].copy()
        elif n == 1:
            df_train = df_known.iloc[:1].reset_index(drop=True)
            df_val = df_known.iloc[0:0].copy()
            df_test_known = df_known.iloc[0:0].copy()
        elif n == 2:
            df_train = df_known.iloc[:1].reset_index(drop=True)
            df_val = df_known.iloc[1:2].reset_index(drop=True)
            df_test_known = df_known.iloc[0:0].copy()
        else:
            n_train = max(1, int(n * 0.70))
            n_val = max(1, int(n * 0.15))
            if n_train + n_val >= n:
                n_train = max(1, n - 2)
                n_val = 1
            df_train = df_known.iloc[:n_train].reset_index(drop=True)
            df_val = df_known.iloc[n_train:n_train + n_val].reset_index(drop=True)
            df_test_known = df_known.iloc[n_train + n_val:].reset_index(drop=True)

        df_test = cls._sort_by_flow_order(
            pd.concat([df_test_known, df_unseen], axis=0, ignore_index=True)
        )
        return df_train, df_val, df_test, df_unseen

    def load_splits(self, max_train_rows: Optional[int] = None) -> Dict[str, pd.DataFrame]:
        dataset_path: Path = self.paths["dataset_path"]
        if self.paths["is_sample_mode"]:
            df_raw = pd.read_parquet(dataset_path)
            self.validate_dataframe_privacy(df_raw, "sample_raw")
            df_train, df_val, df_test, df_unseen = self._split_sample_temporal(df_raw)
        else:
            df_train = pd.read_parquet(dataset_path / "train" / "flows_tabular.parquet")
            df_val = pd.read_parquet(dataset_path / "val" / "flows_tabular.parquet")
            df_test = pd.read_parquet(dataset_path / "test" / "flows_tabular.parquet")
            unseen_path = dataset_path / "test" / "unseen_attack_holdout.parquet"
            if unseen_path.is_file():
                df_unseen = pd.read_parquet(unseen_path)
            elif "attack_family" in df_test.columns:
                df_unseen = df_test[df_test["attack_family"].astype(str).isin(UNSEEN_ATTACK_FAMILIES)].copy()
            else:
                df_unseen = df_test.iloc[0:0].copy()

        if max_train_rows is not None and len(df_train) > max_train_rows:
            df_train = df_train.iloc[:max_train_rows].copy()

        splits = {"train": df_train, "val": df_val, "test": df_test, "unseen_holdout": df_unseen}
        for name, frame in splits.items():
            audit = self.validate_dataframe_privacy(frame, name)
            print(f"[PRIVACY PASS] {audit}")
        return splits


loader = PrivacyAndDataLoader(PATHS)
splits = loader.load_splits(max_train_rows=CONFIG.max_train_rows)

[PRIVACY PASS] {'split': 'train', 'passed_no_l7_audit': True, 'rows': 425771, 'violations': 0}
[PRIVACY PASS] {'split': 'val', 'passed_no_l7_audit': True, 'rows': 91230, 'violations': 0}
[PRIVACY PASS] {'split': 'test', 'passed_no_l7_audit': True, 'rows': 135108, 'violations': 0}
[PRIVACY PASS] {'split': 'unseen_holdout', 'passed_no_l7_audit': True, 'rows': 635, 'violations': 0}


## Cell 3 — Train-Only Feature Sanitization, Scaling & Label Encoding

In [3]:
from sklearn.preprocessing import StandardScaler


class FlowFeatureTransformer:
    """
    Train-only feature sanitizer & scaler (SR-08 anti-leakage):
    1. Casts to float64 column-wise and replaces +inf/-inf (empirical overflow in *_second_bytes) with NaN.
    2. Imputes with Train-only finite medians and clips to Train-only quantile bounds.
    3. Fits StandardScaler strictly on Train (clamping scale_ < 1e-12 -> 1.0) and outputs finite float32 arrays.
    """

    def __init__(
        self,
        feature_cols: Sequence[str] = L3_L4_FEATURE_COLUMNS,
        clip_quantiles: Tuple[float, float] = (0.0, 0.9999),
        output_dtype: str = "float32",
    ) -> None:
        if len(feature_cols) == 0:
            raise ValueError("feature_cols must not be empty.")
        if len(clip_quantiles) != 2 or not (0.0 <= clip_quantiles[0] <= clip_quantiles[1] <= 1.0):
            raise ValueError(f"clip_quantiles must satisfy 0.0 <= low <= high <= 1.0, got {clip_quantiles}")
        if output_dtype not in ("float32", "float64"):
            raise ValueError(f"output_dtype must be 'float32' or 'float64', got '{output_dtype}'")

        self.feature_cols: Tuple[str, ...] = tuple(feature_cols)
        self.clip_quantiles: Tuple[float, float] = (float(clip_quantiles[0]), float(clip_quantiles[1]))
        self.output_dtype: str = output_dtype

        self.scaler_: StandardScaler = StandardScaler()
        self.medians_: Optional[np.ndarray] = None
        self.lower_bounds_: Optional[np.ndarray] = None
        self.upper_bounds_: Optional[np.ndarray] = None
        self.is_fitted_: bool = False

    def _validate_columns(self, df: pd.DataFrame) -> None:
        missing = [c for c in self.feature_cols if c not in df.columns]
        if missing:
            raise ValueError(f"DataFrame is missing required feature columns: {missing}")

    def _extract_float64(self, df: pd.DataFrame) -> np.ndarray:
        self._validate_columns(df)
        cols = list(self.feature_cols)
        arrays = []
        for col in cols:
            series = df[col]
            if pd.api.types.is_numeric_dtype(series):
                arr = series.to_numpy(dtype=np.float64, copy=True)
            else:
                arr = pd.to_numeric(series, errors="coerce").to_numpy(dtype=np.float64, copy=True)
            arrays.append(arr)
        X = np.column_stack(arrays).astype(np.float64, copy=False)
        return np.where(np.isfinite(X), X, np.nan)

    def fit(self, df_train: pd.DataFrame) -> "FlowFeatureTransformer":
        if len(df_train) == 0:
            raise ValueError("Cannot fit FlowFeatureTransformer on an empty DataFrame.")

        X = self._extract_float64(df_train)
        n_features = X.shape[1]
        medians = np.zeros(n_features, dtype=np.float64)
        for idx in range(n_features):
            col_vals = X[:, idx]
            finite_mask = np.isfinite(col_vals)
            medians[idx] = float(np.median(col_vals[finite_mask])) if np.any(finite_mask) else 0.0
        self.medians_ = medians

        nan_mask = np.isnan(X)
        if np.any(nan_mask):
            X = np.where(nan_mask, self.medians_, X)

        q_low, q_high = self.clip_quantiles
        self.lower_bounds_ = np.quantile(X, q_low, axis=0).astype(np.float64)
        self.upper_bounds_ = np.quantile(X, q_high, axis=0).astype(np.float64)
        X = np.clip(X, self.lower_bounds_, self.upper_bounds_)

        self.scaler_ = StandardScaler()
        self.scaler_.fit(X)
        self.scaler_.scale_ = np.where(self.scaler_.scale_ < 1e-12, 1.0, self.scaler_.scale_)
        self.is_fitted_ = True
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        if not self.is_fitted_:
            raise RuntimeError("FlowFeatureTransformer must be fitted on Train before transform().")
        self._validate_columns(df)

        out_dtype = np.dtype(self.output_dtype)
        if len(df) == 0:
            return np.empty((0, len(self.feature_cols)), dtype=out_dtype)

        X = self._extract_float64(df)
        nan_mask = np.isnan(X)
        if np.any(nan_mask):
            X = np.where(nan_mask, self.medians_, X)

        X = np.clip(X, self.lower_bounds_, self.upper_bounds_)
        X = (X - self.scaler_.mean_) / self.scaler_.scale_
        X_out = X.astype(out_dtype, copy=False)
        if not np.all(np.isfinite(X_out)):
            raise ValueError("Non-finite values detected after FlowFeatureTransformer.transform().")
        return X_out

    def fit_transform(self, df_train: pd.DataFrame) -> np.ndarray:
        return self.fit(df_train).transform(df_train)

    def get_state(self) -> Dict[str, Any]:
        if not self.is_fitted_:
            raise RuntimeError("Cannot serialize an unfitted FlowFeatureTransformer.")
        return {
            "feature_cols": list(self.feature_cols),
            "clip_quantiles": [float(self.clip_quantiles[0]), float(self.clip_quantiles[1])],
            "output_dtype": self.output_dtype,
            "medians": [float(x) for x in self.medians_],
            "lower_bounds": [float(x) for x in self.lower_bounds_],
            "upper_bounds": [float(x) for x in self.upper_bounds_],
            "mean": [float(x) for x in self.scaler_.mean_],
            "scale": [float(x) for x in self.scaler_.scale_],
            "var": [float(x) for x in self.scaler_.var_],
            "n_samples_seen": int(self.scaler_.n_samples_seen_),
        }

    @classmethod
    def from_state(cls, state: Dict[str, Any]) -> "FlowFeatureTransformer":
        obj = cls(
            feature_cols=state["feature_cols"],
            clip_quantiles=(float(state["clip_quantiles"][0]), float(state["clip_quantiles"][1])),
            output_dtype=str(state.get("output_dtype", "float32")),
        )
        obj.medians_ = np.asarray(state["medians"], dtype=np.float64)
        obj.lower_bounds_ = np.asarray(state["lower_bounds"], dtype=np.float64)
        obj.upper_bounds_ = np.asarray(state["upper_bounds"], dtype=np.float64)

        scaler = StandardScaler()
        scaler.mean_ = np.asarray(state["mean"], dtype=np.float64)
        scaler.scale_ = np.asarray(state["scale"], dtype=np.float64)
        scaler.scale_ = np.where(scaler.scale_ < 1e-12, 1.0, scaler.scale_)
        scaler.var_ = np.asarray(state["var"], dtype=np.float64)
        scaler.n_features_in_ = len(obj.feature_cols)
        scaler.n_samples_seen_ = int(state.get("n_samples_seen", 0))
        obj.scaler_ = scaler
        obj.is_fitted_ = True
        return obj


class TargetLabelEncoder:
    """Fits known `attack_family` classes on Train; maps unseen or missing labels safely to -1."""

    def __init__(self, unknown_index: int = -1, unknown_label: str = "UNKNOWN_UNSEEN") -> None:
        self.unknown_index: int = int(unknown_index)
        self.unknown_label: str = str(unknown_label)
        self.classes_: List[str] = []
        self.class_to_idx_: Dict[str, int] = {}
        self.idx_to_class_: Dict[int, str] = {}
        self.is_fitted_: bool = False

    def fit(self, y: Any) -> "TargetLabelEncoder":
        values = list(y)
        if len(values) == 0:
            raise ValueError("Cannot fit TargetLabelEncoder on empty labels.")
        unique_classes = sorted({str(v) for v in values if pd.notna(v)})
        if not unique_classes:
            raise ValueError("TargetLabelEncoder found no non-null label classes to fit.")
        self.classes_ = unique_classes
        self.class_to_idx_ = {c: i for i, c in enumerate(self.classes_)}
        self.idx_to_class_ = {i: c for i, c in enumerate(self.classes_)}
        self.is_fitted_ = True
        return self

    def transform(self, y: Any) -> np.ndarray:
        if not self.is_fitted_:
            raise RuntimeError("TargetLabelEncoder must be fitted before calling transform().")
        encoded = [
            self.class_to_idx_.get(str(v), self.unknown_index) if pd.notna(v) else self.unknown_index
            for v in y
        ]
        return np.asarray(encoded, dtype=np.int64)

    def fit_transform(self, y: Any) -> np.ndarray:
        return self.fit(y).transform(y)

    def inverse_transform(self, indices: Sequence[int]) -> List[str]:
        if not self.is_fitted_:
            raise RuntimeError("TargetLabelEncoder must be fitted before calling inverse_transform().")
        return [self.idx_to_class_.get(int(idx), self.unknown_label) for idx in indices]

    def known_mask(self, y_encoded: Sequence[int]) -> np.ndarray:
        return np.asarray(y_encoded, dtype=np.int64) != self.unknown_index

    def get_state(self) -> Dict[str, Any]:
        if not self.is_fitted_:
            raise RuntimeError("Cannot serialize an unfitted TargetLabelEncoder.")
        return {
            "classes": list(self.classes_),
            "unknown_index": self.unknown_index,
            "unknown_label": self.unknown_label,
        }

    @classmethod
    def from_state(cls, state: Dict[str, Any]) -> "TargetLabelEncoder":
        obj = cls(
            unknown_index=int(state.get("unknown_index", -1)),
            unknown_label=str(state.get("unknown_label", "UNKNOWN_UNSEEN")),
        )
        obj.classes_ = [str(c) for c in state["classes"]]
        obj.class_to_idx_ = {c: i for i, c in enumerate(obj.classes_)}
        obj.idx_to_class_ = {i: c for i, c in enumerate(obj.classes_)}
        obj.is_fitted_ = True
        return obj


transformer = FlowFeatureTransformer().fit(splits["train"])
label_encoder = TargetLabelEncoder().fit(splits["train"]["attack_family"])
print(f"Transformer fitted on Train ({len(splits['train']):,} rows, {len(L3_L4_FEATURE_COLUMNS)} features).")
print(f"Known Train attack_family classes ({len(label_encoder.classes_)}): {label_encoder.classes_[:10]}...")

Transformer fitted on Train (425,771 rows, 40 features).
Known Train attack_family classes (19): ['Analysis', 'Backdoor', 'Benign', 'Bot', 'Brute Force', 'DDoS', 'DoS', 'Exploits', 'Fuzzers', 'Generic']...


## Cell 4 — TabularFlowDataset Abstraction

In [4]:
class TabularFlowDataset:
    """Encapsulates sanitized 2D L3/L4 features alongside binary and multi-class targets."""

    def __init__(
        self,
        df: pd.DataFrame,
        transformer: FlowFeatureTransformer,
        label_encoder: TargetLabelEncoder,
    ) -> None:
        self.df = df.reset_index(drop=True)
        self.X: np.ndarray = transformer.transform(self.df)
        self.y_binary: np.ndarray = self.df["label"].to_numpy(dtype=np.int64)
        self.y_multiclass: np.ndarray = label_encoder.transform(self.df["attack_family"])
        self.feature_cols = list(transformer.feature_cols)

    def __len__(self) -> int:
        return len(self.X)


train_ds = TabularFlowDataset(splits["train"], transformer, label_encoder)
val_ds = TabularFlowDataset(splits["val"], transformer, label_encoder)
test_ds = TabularFlowDataset(splits["test"], transformer, label_encoder)
print(f"Datasets ready -> Train: {len(train_ds):,}, Val: {len(val_ds):,}, Test: {len(test_ds):,}")

Datasets ready -> Train: 425,771, Val: 91,230, Test: 135,108


## Cell 5 — Deterministic Rules Baseline & Random Forest Models (OOP)

In [5]:
from sklearn.ensemble import RandomForestClassifier


class RulesBaselineModel:
    """
    Deterministic L3/L4 + optional L5/L6 cryptographic policy baseline emitting bounded scores [0, 1]
    and per-row human-readable reason codes without reading any raw Layer-7 payload.
    """

    def __init__(
        self,
        threshold: float = 0.45,
        syn_flood_pkt_threshold: int = 100,
        high_retransmit_pkts: int = 20,
        extreme_bytes_ratio: float = 100.0,
        extreme_pkts_ratio: float = 50.0,
        high_throughput_bps: float = 1e8,
    ) -> None:
        self.model_name = "week5_rules_baseline"
        self.task_type = "binary"
        self.threshold = float(threshold)
        self.syn_flood_pkt_threshold = int(syn_flood_pkt_threshold)
        self.high_retransmit_pkts = int(high_retransmit_pkts)
        self.extreme_bytes_ratio = float(extreme_bytes_ratio)
        self.extreme_pkts_ratio = float(extreme_pkts_ratio)
        self.high_throughput_bps = float(high_throughput_bps)
        self.is_fitted = True
        self.last_reason_codes_: Optional[List[List[str]]] = None

    def fit(self, X: Any, y: Optional[Any] = None, **kwargs: Any) -> "RulesBaselineModel":
        self.is_fitted = True
        return self

    @staticmethod
    def _col(df: pd.DataFrame, name: str, default: float = 0.0) -> np.ndarray:
        if name in df.columns:
            return (
                pd.to_numeric(df[name], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .fillna(default)
                .to_numpy(dtype=np.float64)
            )
        return np.full(len(df), default, dtype=np.float64)

    def _evaluate_rules(self, df: pd.DataFrame) -> Tuple[np.ndarray, List[List[str]]]:
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"RulesBaselineModel expects a pandas DataFrame, got {type(df).__name__}.")
        n = len(df)
        if n == 0:
            return np.zeros((0,), dtype=np.float64), []

        scores = np.zeros(n, dtype=np.float64)
        reasons: List[List[str]] = [[] for _ in range(n)]

        in_pkts = self._col(df, "in_pkts")
        out_pkts = self._col(df, "out_pkts")
        in_bytes = self._col(df, "in_bytes")
        out_bytes = self._col(df, "out_bytes")
        dur_ms = self._col(df, "flow_duration_ms")
        tcp_flags = self._col(df, "tcp_flags").astype(np.int64)
        client_flags = self._col(df, "client_tcp_flags").astype(np.int64)
        retrans_in = self._col(df, "retransmitted_in_pkts")
        retrans_out = self._col(df, "retransmitted_out_pkts")
        bytes_ratio = self._col(df, "bytes_ratio", 1.0)
        pkts_ratio = self._col(df, "pkts_ratio", 1.0)
        dst_port = self._col(df, "dst_port").astype(np.int64)
        is_ssh = self._col(df, "is_ssh_port")
        src_tp = self._col(df, "src_to_dst_avg_throughput")

        # Rule 1: SYN scan or SYN flood burst
        r1 = ((tcp_flags == 2) | (client_flags == 2)) & ((out_pkts == 0) | (in_pkts >= self.syn_flood_pkt_threshold))
        scores += np.where(r1, 0.45, 0.0)
        for i in np.flatnonzero(r1):
            reasons[i].append("RULE_L4_SYN_SCAN_OR_FLOOD")

        # Rule 2: SSH brute-force / short high-churn connection storm
        r2 = ((is_ssh == 1) | (dst_port == 22) | (dst_port == 2222)) & (
            (in_pkts >= 50) | (retrans_in >= self.high_retransmit_pkts) | ((dur_ms <= 50) & (in_pkts >= 10))
        )
        scores += np.where(r2, 0.50, 0.0)
        for i in np.flatnonzero(r2):
            reasons[i].append("RULE_SSH_BRUTEFORCE_CHURN")

        # Rule 3: Severe TCP retransmission burst
        r3 = (retrans_in + retrans_out) >= self.high_retransmit_pkts
        scores += np.where(r3, 0.35, 0.0)
        for i in np.flatnonzero(r3):
            reasons[i].append("RULE_L4_HIGH_RETRANSMISSION")

        # Rule 4: Extreme flow asymmetry
        r4 = (bytes_ratio >= self.extreme_bytes_ratio) | (pkts_ratio >= self.extreme_pkts_ratio) | (
            (in_bytes >= 50000) & (out_bytes <= 100)
        )
        scores += np.where(r4, 0.40, 0.0)
        for i in np.flatnonzero(r4):
            reasons[i].append("RULE_L3_EXTREME_FLOW_ASYMMETRY")

        # Rule 5: High-throughput burst anomaly
        r5 = (src_tp >= self.high_throughput_bps) & (dur_ms <= 5000)
        scores += np.where(r5, 0.30, 0.0)
        for i in np.flatnonzero(r5):
            reasons[i].append("RULE_L4_HIGH_THROUGHPUT_BURST")

        # Optional L5/L6 hooks (CR-01..CR-04); ECH (`ech_observed`) is strictly never flagged (CR-03)
        if "tls_version" in df.columns:
            tls_ver = df["tls_version"].astype(str).str.upper()
            r_tls = tls_ver.isin({"SSLV2", "SSLV3", "TLSV10", "TLSV11", "TLS1.0", "TLS1.1"}).to_numpy()
            scores += np.where(r_tls, 0.60, 0.0)
            for i in np.flatnonzero(r_tls):
                reasons[i].append("RULE_CR01_DEPRECATED_TLS_VERSION")

        if "cert_status" in df.columns:
            cert_st = df["cert_status"].astype(str).str.lower()
            r_cert = cert_st.isin({"expired", "self_signed", "revoked", "untrusted"}).to_numpy()
            scores += np.where(r_cert, 0.55, 0.0)
            for i in np.flatnonzero(r_cert):
                reasons[i].append("RULE_CR02_CERT_TRUST_ANOMALY")

        if "ssh_kex_class" in df.columns:
            ssh_kex = df["ssh_kex_class"].astype(str).str.lower()
            r_kex = ssh_kex.str.contains("legacy|sha1|group1|weak", na=False).to_numpy()
            scores += np.where(r_kex, 0.55, 0.0)
            for i in np.flatnonzero(r_kex):
                reasons[i].append("RULE_CR04_WEAK_SSH_KEX")

        scores = np.clip(scores, 0.0, 1.0)
        self.last_reason_codes_ = reasons
        return scores, reasons

    def score_samples(self, X: pd.DataFrame, **kwargs: Any) -> np.ndarray:
        scores, _ = self._evaluate_rules(X)
        return scores

    def predict_proba(self, X: pd.DataFrame, **kwargs: Any) -> np.ndarray:
        scores, _ = self._evaluate_rules(X)
        return np.column_stack([1.0 - scores, scores])

    def predict(self, X: pd.DataFrame, **kwargs: Any) -> np.ndarray:
        scores, _ = self._evaluate_rules(X)
        return (scores >= self.threshold).astype(np.int64)

    def explain(self, X: pd.DataFrame) -> List[List[str]]:
        _, reasons = self._evaluate_rules(X)
        return reasons

    def predict_with_reasons(self, df: pd.DataFrame) -> Dict[str, Any]:
        scores, reasons = self._evaluate_rules(df)
        preds = (scores >= self.threshold).astype(np.int64)
        return {"predictions": preds, "scores": scores, "reason_codes": reasons, "threshold": self.threshold}

    def get_metadata(self) -> Dict[str, Any]:
        return {
            "model_name": self.model_name,
            "task_type": self.task_type,
            "deterministic": True,
            "threshold": self.threshold,
            "l7_payload_used": False,
            "ech_treated_as_privacy_feature": True,
        }


class RandomForestThreatModel:
    """OOP Random Forest supporting both binary (`label`) and multi-class (`attack_family`)."""

    def __init__(
        self,
        task: str = "binary",
        n_estimators: int = 100,
        max_depth: Optional[int] = 20,
        min_samples_split: int = 2,
        min_samples_leaf: int = 1,
        class_weight: Any = "balanced_subsample",
        random_state: int = 42,
        n_jobs: int = -1,
    ) -> None:
        if task not in ("binary", "multiclass"):
            raise ValueError(f"Unsupported task '{task}'. Expected 'binary' or 'multiclass'.")
        self.model_name = f"week5_rf_{task}"
        self.task = task
        self.task_type = task
        self.n_estimators = int(n_estimators)
        self.max_depth = max_depth
        self.min_samples_split = int(min_samples_split)
        self.min_samples_leaf = int(min_samples_leaf)
        self.class_weight = class_weight
        self.random_state = int(random_state)
        self.n_jobs = int(n_jobs)
        self.estimator = RandomForestClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf,
            class_weight=self.class_weight,
            random_state=self.random_state,
            n_jobs=self.n_jobs,
        )
        self.n_features_in_: Optional[int] = None
        self.classes_: Optional[np.ndarray] = None
        self.is_fitted: bool = False

    @staticmethod
    def _to_numpy(X: Any) -> np.ndarray:
        arr = np.asarray(X.X if hasattr(X, "X") else X, dtype=np.float32)
        if not np.isfinite(arr).all():
            raise ValueError("Input features contain NaN or Inf. Run FlowFeatureTransformer first.")
        return arr

    def fit(self, X: Any, y: np.ndarray) -> "RandomForestThreatModel":
        X_arr = self._to_numpy(X)
        y_arr = np.asarray(y)
        mask = y_arr >= 0
        X_fit = X_arr[mask] if not np.all(mask) else X_arr
        y_fit = y_arr[mask] if not np.all(mask) else y_arr
        self.estimator.fit(X_fit, y_fit)
        self.n_features_in_ = int(X_fit.shape[1])
        self.classes_ = np.asarray(self.estimator.classes_)
        self.is_fitted = True
        return self

    def predict(self, X: Any) -> np.ndarray:
        if not self.is_fitted:
            raise RuntimeError("RandomForestThreatModel must be fitted before predict().")
        return np.asarray(self.estimator.predict(self._to_numpy(X)))

    def predict_proba(self, X: Any) -> np.ndarray:
        if not self.is_fitted:
            raise RuntimeError("RandomForestThreatModel must be fitted before predict_proba().")
        return np.asarray(self.estimator.predict_proba(self._to_numpy(X)), dtype=np.float64)

    def get_feature_importance(self, feature_names: Sequence[str]) -> Dict[str, float]:
        if not self.is_fitted:
            raise RuntimeError("Model must be fitted before querying feature importance.")
        imp = np.asarray(self.estimator.feature_importances_, dtype=np.float64)
        imp = imp / imp.sum() if imp.sum() > 0 else np.full(len(imp), 1.0 / len(imp))
        return dict(sorted(zip(feature_names, imp.tolist()), key=lambda kv: kv[1], reverse=True))

    def get_metadata(self) -> Dict[str, Any]:
        return {
            "model_name": self.model_name,
            "task": self.task,
            "task_type": self.task_type,
            "n_estimators": self.n_estimators,
            "max_depth": self.max_depth,
            "min_samples_split": self.min_samples_split,
            "min_samples_leaf": self.min_samples_leaf,
            "class_weight": str(self.class_weight),
            "random_state": self.random_state,
            "n_features_in": self.n_features_in_,
            "num_classes": int(len(self.classes_)) if self.classes_ is not None else 0,
            "is_fitted": self.is_fitted,
        }

    @classmethod
    def from_checkpoint(cls, payload: Any) -> "RandomForestThreatModel":
        if isinstance(payload, cls):
            return payload
        if isinstance(payload, dict):
            if "model" in payload and isinstance(payload["model"], cls):
                return payload["model"]
            if "estimator" in payload:
                est = payload["estimator"]
                state = payload.get("model_state") or payload.get("state") or {}
                task = str(state.get("task", state.get("task_type", "binary")))
                if task not in ("binary", "multiclass"):
                    task = "multiclass" if len(getattr(est, "classes_", [])) > 2 else "binary"
                obj = cls(
                    task=task,
                    n_estimators=int(state.get("n_estimators", getattr(est, "n_estimators", 100))),
                    max_depth=state.get("max_depth", getattr(est, "max_depth", 20)),
                    min_samples_split=int(state.get("min_samples_split", getattr(est, "min_samples_split", 2))),
                    min_samples_leaf=int(state.get("min_samples_leaf", getattr(est, "min_samples_leaf", 1))),
                    class_weight=getattr(est, "class_weight", "balanced_subsample"),
                    random_state=int(state.get("random_state", getattr(est, "random_state", 42) or 42)),
                    n_jobs=int(getattr(est, "n_jobs", state.get("n_jobs", -1)) or -1),
                )
                obj.estimator = est
                n_feat = state.get("n_features_in")
                obj.n_features_in_ = int(n_feat) if n_feat is not None else getattr(est, "n_features_in_", None)
                obj.classes_ = np.asarray(est.classes_) if hasattr(est, "classes_") else None
                obj.is_fitted = bool(state.get("is_fitted", hasattr(est, "classes_")))
                return obj
        raise TypeError(f"Unsupported RandomForest checkpoint payload type: {type(payload).__name__}")

## Cell 6 — Trainer & Evaluator Classes

In [6]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


class Week5Trainer:
    """Trains binary and multi-class Random Forest models and evaluates Validation F1."""

    def __init__(self, n_estimators: int = 100, max_depth: Optional[int] = 20, seed: int = 42) -> None:
        self.rf_binary = RandomForestThreatModel(task="binary", n_estimators=n_estimators, max_depth=max_depth, random_state=seed)
        self.rf_multiclass = RandomForestThreatModel(task="multiclass", n_estimators=n_estimators, max_depth=max_depth, random_state=seed)

    def train_all(
        self,
        train_ds: TabularFlowDataset,
        val_ds: Optional[TabularFlowDataset] = None,
    ) -> Dict[str, Any]:
        t0 = time.perf_counter()
        self.rf_binary.fit(train_ds.X, train_ds.y_binary)
        t_bin = time.perf_counter() - t0

        t1 = time.perf_counter()
        self.rf_multiclass.fit(train_ds.X, train_ds.y_multiclass)
        t_mc = time.perf_counter() - t1

        summary: Dict[str, Any] = {"binary_fit_seconds": t_bin, "multiclass_fit_seconds": t_mc}
        if val_ds is not None and len(val_ds) > 0:
            val_bin_preds = self.rf_binary.predict(val_ds.X)
            summary["val_binary_f1"] = float(f1_score(val_ds.y_binary, val_bin_preds, zero_division=0))
            known_val = val_ds.y_multiclass >= 0
            if np.any(known_val):
                val_mc_preds = self.rf_multiclass.predict(val_ds.X[known_val])
                summary["val_multiclass_macro_f1"] = float(
                    f1_score(val_ds.y_multiclass[known_val], val_mc_preds, average="macro", zero_division=0)
                )
        return summary


class Week5Evaluator:
    """Computes binary metrics, multi-class metrics, latency, confusion matrix, and Rules vs RF comparison."""

    @staticmethod
    def evaluate_binary(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray) -> Dict[str, Any]:
        if y_score.ndim == 2:
            y_score = y_score[:, 1]
        roc = float(roc_auc_score(y_true, y_score)) if len(np.unique(y_true)) > 1 else 0.5
        pr = float(average_precision_score(y_true, y_score)) if len(np.unique(y_true)) > 1 else float(np.mean(y_true == 1))
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])
        return {
            "roc_auc": roc,
            "pr_auc": pr,
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "fpr": float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0,
            "confusion_matrix": cm.tolist(),
            "n_samples": int(len(y_true)),
        }

    @staticmethod
    def evaluate_multiclass(y_true: np.ndarray, y_pred: np.ndarray, class_names: Sequence[str]) -> Dict[str, Any]:
        known = y_true >= 0
        n_unseen = int((~known).sum())
        yt, yp = y_true[known], y_pred[known]
        if len(yt) == 0:
            return {
                "macro_f1": 0.0,
                "micro_f1": 0.0,
                "per_class_recall": {},
                "confusion_matrix": [],
                "n_known_samples": 0,
                "n_unseen_excluded_from_mc": n_unseen,
            }
        labels = sorted(set(yt.tolist()) | set(yp.tolist()))
        recalls = recall_score(yt, yp, labels=labels, average=None, zero_division=0)
        per_class_recall = {
            (class_names[l] if 0 <= l < len(class_names) else f"class_{l}"): float(recalls[i])
            for i, l in enumerate(labels)
        }
        return {
            "macro_f1": float(f1_score(yt, yp, average="macro", zero_division=0)),
            "micro_f1": float(f1_score(yt, yp, average="micro", zero_division=0)),
            "per_class_recall": per_class_recall,
            "confusion_matrix": confusion_matrix(yt, yp, labels=labels).tolist(),
            "n_known_samples": int(len(yt)),
            "n_unseen_excluded_from_mc": n_unseen,
        }

    @staticmethod
    def compare_baselines(rules_metrics: Dict[str, Any], rf_metrics: Dict[str, Any]) -> pd.DataFrame:
        keys = ("roc_auc", "pr_auc", "f1", "precision", "recall", "fpr", "inference_latency_ms_per_flow")
        return pd.DataFrame([
            {"Model": "Rules_Baseline", **{k: rules_metrics.get(k) for k in keys}},
            {"Model": "RandomForest_Binary", **{k: rf_metrics.get(k) for k in keys}},
        ])


class CheckpointManager:
    """Saves/reloads `.joblib` artifacts + sanitized `_metadata.json` provenance (`allow_nan=False`)."""

    def __init__(self, ckpt_dir: Path) -> None:
        self.ckpt_dir = Path(ckpt_dir)
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)

    @classmethod
    def _sanitize_json(cls, obj: Any) -> Any:
        if obj is None or isinstance(obj, (bool, str, int)):
            return obj
        if isinstance(obj, float):
            return obj if math.isfinite(obj) else None
        if isinstance(obj, Path):
            return str(obj)
        if isinstance(obj, np.bool_):
            return bool(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            val = float(obj)
            return val if math.isfinite(val) else None
        if isinstance(obj, np.ndarray):
            return cls._sanitize_json(obj.tolist())
        if isinstance(obj, Mapping):
            return {str(k): cls._sanitize_json(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple, set, frozenset)):
            return [cls._sanitize_json(v) for v in obj]
        return str(obj)

    def save(self, name: str, payload: Dict[str, Any], metadata: Dict[str, Any]) -> Tuple[Path, Path]:
        art_path = self.ckpt_dir / f"{name}.joblib"
        meta_path = self.ckpt_dir / f"{name}_metadata.json"
        joblib.dump(payload, art_path, compress=3)
        clean_meta = self._sanitize_json(metadata)
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(clean_meta, f, indent=2, ensure_ascii=False, allow_nan=False)
        return art_path, meta_path

    def load(self, name: str) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        art_path = self.ckpt_dir / f"{name}.joblib"
        meta_path = self.ckpt_dir / f"{name}_metadata.json"
        payload = joblib.load(art_path)
        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)
        return payload, meta

## Cell 7 — Training, Evaluation, Feature Importance & Checkpointing

In [7]:
# Auto-scale estimators for lightweight sample smoke test vs full Kaggle GPU run
n_trees = 30 if PATHS["is_sample_mode"] else 150
trainer = Week5Trainer(n_estimators=n_trees, max_depth=20, seed=CONFIG.seed)
timings = trainer.train_all(train_ds, val_ds)
print("Training timings & Val F1:", timings)

evaluator = Week5Evaluator()

# 1. Deterministic Rules Baseline
t_r0 = time.perf_counter()
rules_model = RulesBaselineModel(threshold=0.45)
rules_out = rules_model.predict_with_reasons(test_ds.df)
t_rules = time.perf_counter() - t_r0
rules_metrics = evaluator.evaluate_binary(test_ds.y_binary, rules_out["predictions"], rules_out["scores"])
rules_metrics["inference_latency_ms_per_flow"] = float((t_rules / max(1, len(test_ds))) * 1000.0)

# 2. Random Forest Binary
t_b0 = time.perf_counter()
rf_bin_probs = trainer.rf_binary.predict_proba(test_ds.X)
t_rf_bin = time.perf_counter() - t_b0
rf_bin_preds = (rf_bin_probs[:, 1] >= 0.5).astype(np.int64) if rf_bin_probs.shape[1] > 1 else trainer.rf_binary.predict(test_ds.X)
rf_bin_metrics = evaluator.evaluate_binary(test_ds.y_binary, rf_bin_preds, rf_bin_probs)
rf_bin_metrics["inference_latency_ms_per_flow"] = float((t_rf_bin / max(1, len(test_ds))) * 1000.0)

# 3. Random Forest Multi-Class (`attack_family`)
t_m0 = time.perf_counter()
rf_mc_preds = trainer.rf_multiclass.predict(test_ds.X)
t_rf_mc = time.perf_counter() - t_m0
rf_mc_metrics = evaluator.evaluate_multiclass(test_ds.y_multiclass, rf_mc_preds, label_encoder.classes_)
rf_mc_metrics["inference_latency_ms_per_flow"] = float((t_rf_mc / max(1, len(test_ds))) * 1000.0)

# 4. Comparison Table & Top-15 Feature Importance
comparison_df = evaluator.compare_baselines(rules_metrics, rf_bin_metrics)
print("\n=== RULES vs RANDOM FOREST (BINARY) COMPARISON ===")
print(comparison_df.to_string(index=False))

print("\n=== RANDOM FOREST MULTI-CLASS METRICS ===")
print(json.dumps({
    "macro_f1": rf_mc_metrics["macro_f1"],
    "micro_f1": rf_mc_metrics["micro_f1"],
    "n_unseen_excluded_from_mc": rf_mc_metrics["n_unseen_excluded_from_mc"],
    "per_class_recall": rf_mc_metrics["per_class_recall"],
}, indent=2))
print("Confusion Matrix (Binary):", rf_bin_metrics["confusion_matrix"])

fi = trainer.rf_binary.get_feature_importance(L3_L4_FEATURE_COLUMNS)
print("\n=== TOP 10 FEATURE IMPORTANCE (RANDOM FOREST) ===")
for k, v in list(fi.items())[:10]:
    print(f"  {k:30s}: {v:.4f}")

# 5. Save cross-environment checkpoints (raw sklearn estimator + JSON state, never __main__ wrapper) & Verify Reload Parity
ckpt_mgr = CheckpointManager(PATHS["checkpoint_dir"])
meta_common = {
    "seed": CONFIG.seed,
    "versions": VERSIONS,
    "environment": VERSIONS,
    "feature_columns": L3_L4_FEATURE_COLUMNS,
    "dataset_mode": PATHS["mode"],
    "val_summary": timings,
    "saved_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}
art_bin, meta_bin = ckpt_mgr.save(
    "week5_rf_binary",
    {
        "estimator": trainer.rf_binary.estimator,
        "model_state": trainer.rf_binary.get_metadata(),
        "transformer_state": transformer.get_state(),
    },
    {**meta_common, "metrics": rf_bin_metrics, "rules_comparison": rules_metrics, "top_features": dict(list(fi.items())[:15])},
)
art_mc, meta_mc = ckpt_mgr.save(
    "week5_rf_multiclass",
    {
        "estimator": trainer.rf_multiclass.estimator,
        "model_state": trainer.rf_multiclass.get_metadata(),
        "transformer_state": transformer.get_state(),
        "label_encoder_state": label_encoder.get_state(),
        "classes": label_encoder.classes_,
    },
    {**meta_common, "metrics": rf_mc_metrics},
)

reloaded_bin_payload, _ = ckpt_mgr.load("week5_rf_binary")
reloaded_rf_bin = RandomForestThreatModel.from_checkpoint(reloaded_bin_payload)
reloaded_tf = FlowFeatureTransformer.from_state(reloaded_bin_payload["transformer_state"])
assert np.allclose(rf_bin_probs, reloaded_rf_bin.predict_proba(reloaded_tf.transform(test_ds.df)), atol=1e-6)
print(f"\n[VERIFIED] Saved & reloaded cross-env checkpoints: {art_bin.name}, {art_mc.name}")

Training timings & Val F1: {'binary_fit_seconds': 48.422975699999995, 'multiclass_fit_seconds': 55.31532813000001, 'val_binary_f1': 0.9962085652949095, 'val_multiclass_macro_f1': 0.9043773461339211}

=== RULES vs RANDOM FOREST (BINARY) COMPARISON ===
              Model  roc_auc   pr_auc       f1  precision   recall      fpr  inference_latency_ms_per_flow
     Rules_Baseline 0.587412 0.796450 0.294383   0.930296 0.174858 0.040075                       0.000380
RandomForest_Binary 0.999503 0.999848 0.996979   0.998788 0.995178 0.003695                       0.005163

=== RANDOM FOREST MULTI-CLASS METRICS ===
{
  "macro_f1": 0.8466145912771319,
  "micro_f1": 0.9492611899786574,
  "n_unseen_excluded_from_mc": 635,
  "per_class_recall": {
    "Analysis": 0.9102244389027432,
    "Backdoor": 0.7037533512064343,
    "Benign": 0.9928803172314348,
    "Bot": 1.0,
    "Brute Force": 0.9985207100591716,
    "DDoS": 0.9887424085320693,
    "DoS": 0.9505697794879384,
    "Exploits": 0.8773506059339